In [1]:
# Save total mortality by region for all ensemble members

In [2]:
import os
import xarray as xr
import numpy as np

In [3]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/region/"

in_file = "GBD_Region_Masks_0.10.nc"
in_path = os.path.join(MASK_DIR, in_file)
region_mask = xr.open_dataarray(in_path)

In [ ]:
# === Path config ===
MORTALITY_DIR = "/glade/work/awells/air_quality/CESM/mortality/"
SCENARIOS = ["ARISE", "SSP245"]

# === Main loop ===
for scenario in SCENARIOS:
    ensembles = []
    for ens_num in range(1, 11):
        print(f"Processing {scenario}, Ensemble {ens_num:02d}")
        if scenario == "ARISE":
            dates = "2035-2068"
        elif scenario == "SSP245":
            dates = "2020-2068"

        in_file = f"Mortality_CESM2_{scenario}_{ens_num:02d}_{dates}.nc"
        in_path = os.path.join(MORTALITY_DIR, in_file)

        if not os.path.exists(in_path):
            print(f"Missing: {in_path}")
            continue
        da = xr.open_dataarray(in_path)

        regions = []
        # Loop over regions and sum the mortality for each region
        for i in range(len(region_mask.region)):
            mask = region_mask.isel(region=i)
            # mean mortality of region (based on BMR mean)
            M_region = (xr.where(mask == 1,
                                 da.sel(quantile="mean"),
                                 np.nan)).sum(dim=("lat", "lon"))
            regions.append(M_region.drop_vars("region", errors='ignore'))
        mortality_region = xr.concat(regions,
                                     dim=xr.DataArray(region_mask["region"],
                                                      dims="region",
                                                      name="region"))

        ensembles.append(mortality_region)

    regions_ens = xr.concat(ensembles,
                            dim=xr.DataArray(np.arange(1, len(ensembles)+1),
                                             dims="ensemble", name="ensemble"))

    regions_ens.attrs["description"] = ("Region level mean Mortality - "
                                        "scripts by A.F. Wells (2025)")
    regions_ens.attrs["scenario"] = scenario

    out_file = f"Mortality_Regional_sum_CESM2_{scenario}_{dates}.nc"
    out_path = os.path.join(MORTALITY_DIR, out_file)
    print(f"Saving regional mortality to {out_path}")
    regions_ens.to_netcdf(out_path)

Processing ARISE, Ensemble 01
Processing ARISE, Ensemble 02
Processing ARISE, Ensemble 03
Processing ARISE, Ensemble 04
Processing ARISE, Ensemble 05
Processing ARISE, Ensemble 06
Processing ARISE, Ensemble 07
Processing ARISE, Ensemble 08
Processing ARISE, Ensemble 09
Processing ARISE, Ensemble 10
Saving regional mortality to /glade/work/awells/air_quality/CESM/mortality/Mortality_Regional_sum_CESM2_ARISE_2035-2068.nc
Processing SSP245, Ensemble 01
Processing SSP245, Ensemble 02
Processing SSP245, Ensemble 03
Processing SSP245, Ensemble 04
Processing SSP245, Ensemble 05
Processing SSP245, Ensemble 06
Processing SSP245, Ensemble 07
Processing SSP245, Ensemble 08
